In [ ]:
import pyarrow.parquet as pq
import polars as pl
import numpy as np
from tqdm.auto import tqdm
import gc

In [ ]:
TEST_PATH = '/kaggle/input/datasets/b22dckh072/file05/test_interactions.parquet'
FINAL_TOP100_PATH = '/kaggle/input/datasets/b22dckh072/file05/top100_final_recommendations.parquet' 
ITEM_MAPPING_PATH = '/kaggle/input/datasets/b22dckh072/file05/item_mapping.parquet'

In [ ]:
print("Đang nạp tập Test và tạo danh sách đối chiếu...")
df_test = pl.read_parquet(TEST_PATH)
truth_df = df_test.group_by('mapped_user_id').agg(pl.col('mapped_item_id').alias('true_items'))
n_valid = truth_df.height

print(f"Số lượng người dùng hợp lệ trong tập Test: {n_valid:,}")
print("Đang tính toán chỉ số bằng luồng dữ liệu (Streaming Evaluation)...")

FINAL_TOP100_PATH = '/kaggle/input/datasets/b22dckh072/file05/top100_final_recommendations.parquet'
pf = pq.ParquetFile(FINAL_TOP100_PATH)

In [ ]:
print("Đang chuẩn bị tập Test ")
truth_df = truth_df.filter((pl.col('mapped_user_id') % 2) != 0)
n_valid = truth_df.select('mapped_user_id').n_unique()
print(f"Số lượng người dùng trong tập Test: {n_valid:,}")
reader = pf.iter_batches(batch_size=5000000, columns=['mapped_user_id', 'mapped_item_id', 'score'])

hr_sum = 0
ndcg_sum = 0.0
rc100_sum = 0  
buffer_df = pl.DataFrame()

for batch in tqdm(reader, desc="Evaluating Chunks"):
    chunk = pl.from_arrow(batch)
    
    if buffer_df.height > 0:
        chunk = pl.concat([buffer_df, chunk])
        
    last_user = chunk.get_column('mapped_user_id')[-1]
    
    completed = chunk.filter(pl.col('mapped_user_id') != last_user)
    buffer_df = chunk.filter(pl.col('mapped_user_id') == last_user)
    
    if completed.height > 0:
        completed = completed.with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('mapped_user_id').alias('rank')
        )
        
        top100 = completed.filter(pl.col('rank') <= 100)
        eval_df_100 = top100.join(truth_df, on='mapped_user_id', how='inner')
        
        hits_100 = eval_df_100.filter(pl.col('true_items').list.contains(pl.col('mapped_item_id')))
        
        if hits_100.height > 0:
            rc100_sum += hits_100.select('mapped_user_id').n_unique()
            
            hits_10 = hits_100.filter(pl.col('rank') <= 10)
            if hits_10.height > 0:
                hr_sum += hits_10.select('mapped_user_id').n_unique()
                ndcg = hits_10.with_columns((1.0 / np.log2(pl.col('rank') + 1)).alias('ndcg_val'))
                ndcg_sum += ndcg.select(pl.col('ndcg_val').sum()).item()
                
        del top100, eval_df_100, hits_100
        
    del chunk, completed
    gc.collect()

if buffer_df.height > 0:
    buffer_df = buffer_df.with_columns(
        pl.col('score').rank(method='ordinal', descending=True).over('mapped_user_id').alias('rank')
    )
    top100 = buffer_df.filter(pl.col('rank') <= 100)
    eval_df_100 = top100.join(truth_df, on='mapped_user_id', how='inner')
    hits_100 = eval_df_100.filter(pl.col('true_items').list.contains(pl.col('mapped_item_id')))
    
    if hits_100.height > 0:
        rc100_sum += hits_100.select('mapped_user_id').n_unique()
        
        hits_10 = hits_100.filter(pl.col('rank') <= 10)
        if hits_10.height > 0:
            hr_sum += hits_10.select('mapped_user_id').n_unique()
            ndcg = hits_10.with_columns((1.0 / np.log2(pl.col('rank') + 1)).alias('ndcg_val'))
            ndcg_sum += ndcg.select(pl.col('ndcg_val').sum()).item()

HR10 = hr_sum / n_valid
NDCG10 = ndcg_sum / n_valid
RC100 = rc100_sum / n_valid

print("=======================================")
print(f"KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH XGBOOST")
print(f"Tập người dùng hợp lệ (Lẻ): {n_valid:,}")
print(f"Recall @ 100 (RC@100):      {RC100:.4f}")
print(f"Hit Rate @ 10 (HR@10):      {HR10:.4f}")
print(f"NDCG @ 10:                  {NDCG10:.4f}")
print("=======================================")

In [ ]:
print("Đang thiết lập luồng dịch ID ")
lf_mapping = pl.scan_parquet(ITEM_MAPPING_PATH)
lf_final = pl.scan_parquet(FINAL_TOP100_PATH)

print("Đang xử lý luồng cắt Top 10 và Join dữ liệu...")
lf_readable_recs = (
    lf_final
    .with_columns(pl.col('score').rank(method='ordinal', descending=True).over('mapped_user_id').alias('rank'))
    .filter(pl.col('rank') <= 10)
    .join(lf_mapping, on='mapped_item_id', how='left')
    .select(['mapped_user_id', 'parent_asin', 'score', 'rank'])
)

print("Đang thực thi và xuất file (Streaming Mode)...")
FINAL_CSV_PATH = '/kaggle/working/final_top10_recommendations.csv'
final_readable_recs = lf_readable_recs.collect(engine="streaming")

final_readable_recs.write_csv(FINAL_CSV_PATH)

print(f"Đã xuất danh sách gợi ý thực tế tại: {FINAL_CSV_PATH}")
print("\nXem thử 5 gợi ý đầu tiên:")
print(final_readable_recs.head(5))
del final_readable_recs
gc.collect()